In [1]:
import s3fs
import xarray
import pandas as pd

## Open table with NHD COMIDs (reach codes/feature ids) of Interest

In [2]:
fn_in = r"C:\Users\khafen\DOI\CDI - Toil Working Group - CDI_TOIL\CDI\data\network\nhd_nwm_matchup.csv"
df_nhd = pd.read_csv(fn_in)
df_nhd

,NHDPlusID,GridCode,NWM_ID
0,55000900271647,38585,23773423
1,55000900133422,43487,23773961
2,55000900167055,35844,23773421
3,55000900201198,38546,23773415
4,55000900237118,42194,23774053
5,55000900133417,37005,23773417
6,55000900029615,206189,23773987
7,55000900237121,36367,23773419
8,55000900062365,42684,23774027
9,55000900235245,37875,23773413


## Load the zarr store from AWS
The chrtout.zarr contains streamflow data
- Info about the AWS zarr store is here: [https://registry.opendata.aws/nwm-archive/](https://registry.opendata.aws/nwm-archive/)
- Documentation about the model outputs in the zarr store is here: [https://github.com/NOAA-Big-Data-Program/bdp-data-docs/blob/main/nwm/README.md](https://github.com/NOAA-Big-Data-Program/bdp-data-docs/blob/main/nwm/README.md)

In [3]:
s3_path = 's3://noaa-nwm-retrospective-2-1-zarr-pds/chrtout.zarr'

# Connect to S3
s3 = s3fs.S3FileSystem(anon=True)
store = s3fs.S3Map(root=s3_path, s3=s3, check=False)

# load the dataset
ds = xarray.open_zarr(store=store)
ds

<xarray.Dataset>
Dimensions:     (feature_id: 2776738, time: 367439)
Coordinates:
    elevation   (feature_id) float32 ...
  * feature_id  (feature_id) int32 101 179 181 ... 1180001803 1180001804
    gage_id     (feature_id) |S15 ...
    latitude    (feature_id) float32 ...
    longitude   (feature_id) float32 ...
    order       (feature_id) int32 ...
  * time        (time) datetime64[ns] 1979-02-01T01:00:00 ... 2020-12-31T23:0...
Data variables:
    crs         |S1 ...
    streamflow  (time, feature_id) float64 ...
    velocity    (time, feature_id) float64 ...
Attributes:
    TITLE:                OUTPUT FROM WRF-Hydro v5.2.0-beta2
    code_version:         v5.2.0-beta2
    featureType:          timeSeries
    model_configuration:  retrospective
    proj4:                +proj=lcc +units=m +a=6370000.0 +b=6370000.0 +lat_1...

## Get data for reaches in HJ Andrews experimental forest
The NWM_ID column corresponds to NHD COMID, which corresponds to NWM feature_id.

In [4]:
id = df_nhd['NWM_ID'].values[0]
# ds_id = ds.where(ds.feature_id == id, drop=True)
ds_id = ds.sel(feature_id=df_nhd['NWM_ID'].values)
streamflow_id = ds_id.streamflow.load()
streamflow_id

<xarray.DataArray 'streamflow' (time: 367439, feature_id: 11)>
array([[0.32999999, 0.11      , 0.64999999, ..., 0.08      , 2.22999995,
        2.61999994],
       [0.32999999, 0.11      , 0.64999999, ..., 0.08      , 2.21999995,
        2.61999994],
       [0.32999999, 0.11      , 0.64999999, ..., 0.08      , 2.21999995,
        2.60999994],
       ...,
       [0.51999999, 0.13      , 1.05999998, ..., 0.09      , 3.54999992,
        4.12999991],
       [0.51999999, 0.13      , 1.02999998, ..., 0.09      , 3.40999992,
        4.01999991],
       [0.50999999, 0.13      , 1.02999998, ..., 0.09      , 3.43999992,
        4.01999991]])
Coordinates:
    elevation   (feature_id) float32 1.202e+03 910.8 728.7 ... 545.4 497.2
  * feature_id  (feature_id) int32 23773423 23773961 ... 23773413 23773411
    gage_id     (feature_id) |S15 b'               ' ... b'       14161500'
    latitude    (feature_id) float32 44.26 44.26 44.24 ... 44.22 44.23 44.22
    longitude   (feature_id) float32 -122.2 -122.2 -122.2 ... -122.2 -122.2
    order       (feature_id) int32 1 1 2 2 1 2 1 1 1 3 3
  * time        (time) datetime64[ns] 1979-02-01T01:00:00 ... 2020-12-31T23:0...
Attributes:
    grid_mapping:  crs
    long_name:     River Flow
    units:         m3 s-1

## Save streamflow to CSV

In [32]:
fn_streamflow = r'C:\Users\khafen\DOI\CDI - Toil Working Group - CDI_TOIL\CDI\data\nwm\streamflow_hja_hourly_cms_1979-2020.csv'
# streamflow_df = streamflow_id.squeeze('feature_id').to_dataframe()
streamflow_df = streamflow_id.to_dataframe()
streamflow_df.to_csv(fn_streamflow, index=False)
streamflow_df.head()

elevation             gage_id   latitude  \
time                feature_id                                               
1979-02-01 01:00:00 23773423    1202.150024  b'               '  44.261803   
                    23773961     910.760010  b'               '  44.258801   
                    23773421     728.690002  b'               '  44.242271   
                    23773415     669.710022  b'               '  44.232731   
                    23774053    1366.819946  b'               '  44.211750   

                                 longitude  order  streamflow  
time                feature_id                                 
1979-02-01 01:00:00 23773423   -122.162674      1        0.33  
                    23773961   -122.180847      1        0.11  
                    23773421   -122.196320      2        0.65  
                    23773415   -122.187981      2        1.41  
                    23774053   -122.154297      1        0.30

## Resample from hourly to daily average

In [42]:
fn_streamflow_day = r'C:\Users\khafen\DOI\CDI - Toil Working Group - CDI_TOIL\CDI\data\nwm\streamflow_hja_daily_cms_1979-2020.csv'
streamflow_reset = streamflow_df.reset_index()
streamflow_reset.index = pd.DatetimeIndex(streamflow_reset.time)
streamflow_df_day = streamflow_reset.groupby('feature_id', as_index=False).resample('D').mean().reset_index()
streamflow_df_day[['time', 'feature_id', 'elevation', 'order', 'streamflow']].to_csv(fn_streamflow_day, index=False)
streamflow_df_day.head(n=30)

# streamflow_df_day = streamflow_df.resample('D').mean()
# streamflow_df_day = streamflow_df.groupby(pd.Grouper(freq='1D', level=0))
# streamflow_df_day.to_csv(fn_streamflow_day, index=False)
# streamflow_df_day.head(n=10)

C:\Users\khafen\AppData\Local\Temp\1\ipykernel_8992\2974782838.py:4: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  streamflow_df_day = streamflow_reset.groupby('feature_id', as_index=False).resample('D').mean().reset_index()


,level_0,time,feature_id,elevation,latitude,longitude,order,streamflow
0,0,1979-02-01,23773411.0,497.179993,44.21907,-122.244797,3.0,2.567391
1,0,1979-02-02,23773411.0,497.180023,44.21907,-122.244789,3.0,2.457500
2,0,1979-02-03,23773411.0,497.180023,44.21907,-122.244789,3.0,2.353333
3,0,1979-02-04,23773411.0,497.180023,44.21907,-122.244789,3.0,2.256250
4,0,1979-02-05,23773411.0,497.180023,44.21907,-122.244789,3.0,2.165833
5,0,1979-02-06,23773411.0,497.180023,44.21907,-122.244789,3.0,2.081667
6,0,1979-02-07,23773411.0,497.180023,44.21907,-122.244789,3.0,2.014167
7,0,1979-02-08,23773411.0,497.180023,44.21907,-122.244789,3.0,1.933750
8,0,1979-02-09,23773411.0,497.180023,44.21907,-122.244789,3.0,2.041667
9,0,1979-02-10,23773411.0,497.180023,44.21907,-122.244789,3.0,6.583333
